# 🧹 MozDev — Script Automation Part 0.1 · Limpeza de Dados

**Objectivo:** Detectar e corrigir os 3 tipos de erros injectados nos ficheiros Excel:

| # | Tipo | Descrição | Função SQL |
|---|------|-----------|------------|
| 1 | `NEGATIVO` | Valores numéricos negativos | `ABS()` |
| 2 | `ESPACOS` | Espaços extra em strings de pagamento | `TRIM()` |
| 3 | `DATA_ERRADA` | Datetime em vez de date pura | `DATE()` |

> 🔵 Azul = input fixo · 🖤 Preto = fórmula · 🟢 Verde = ligação entre folhas

## ⚙️ Setup — instalar dependências e ligar ao Drive

In [ ]:
# Célula 1 — instalar dependências
!pip install jupysql pandas openpyxl --quiet

In [ ]:
# Célula 2 — importar bibliotecas
import pandas as pd
import sqlite3
from google.colab import drive
import os

In [ ]:
# Célula 3 — montar Google Drive e definir pasta de trabalho
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/MozDev_AI_Automation_Scripts/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Drive montado em: {OUTPUT_DIR}")

In [ ]:
# Célula 4 — navegar para a pasta de trabalho
%cd /content/drive/MyDrive/MozDev_AI_Automation_Scripts/
%ls

## 🗄️ Criar base de dados SQLite a partir dos ficheiros Excel

In [ ]:
# Célula 5 — carregar os 3 ficheiros Excel COM ERROS para a base de dados
# Nota: header=1 porque a linha 1 é o título decorativo e a linha 2 tem os cabeçalhos reais

conn = sqlite3.connect("mozchapa100.db")

pd.read_excel("MozChapa100_Segmentos_ERROS.xlsx", header=1).to_sql("segmentos", conn, if_exists="replace", index=False)
pd.read_excel("MozChapa100_Vendas_ERROS.xlsx",    header=1).to_sql("vendas",    conn, if_exists="replace", index=False)
pd.read_excel("MozChapa100_Rotas_ERROS.xlsx",     header=1).to_sql("rotas",     conn, if_exists="replace", index=False)

print("✅ DB criada com 3 tabelas: segmentos | vendas | rotas")

In [ ]:
# Célula 6 — activar SQL magic (permite escrever SQL directamente nas células)
%load_ext sql
%sql sqlite:///mozchapa100.db

## 🔗 Criar tabela analítica — JOIN das 3 tabelas

Antes de limpar, precisamos de juntar `vendas`, `segmentos` e `rotas` numa só tabela de análise.
Esta tabela ainda **tem os erros** — vamos detectá-los a seguir.

> **Porquê `CREATE TABLE` e não `CREATE VIEW`?**  
> Em SQLite, `CREATE VIEW` é só uma query guardada — não armazena dados.  
> `CREATE TABLE AS SELECT` cria uma cópia física dos dados, mais rápida para consultar e limpar.

In [ ]:
%%sql
-- Célula 7 — criar tabela analítica com JOIN das 3 tabelas
-- Esta tabela ainda contém os erros originais (negativos, espaços, datas)

CREATE TABLE MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS AS
SELECT
    v.Data                              AS dia,
    s.Tipo_Passageiro                   AS segmento,
    (r.Origem || ' - ' || r.Destino)   AS rota,
    v.Forma_Pagamento,

    -- métricas de vendas
    SUM(v.Bilhetes_Vendidos)                        AS Bilhetes_Vendidos,
    AVG(v.Preco_Unitario_MZN)                       AS Preco_Unitario_MZN,
    SUM(v.Bilhetes_Vendidos * v.Preco_Unitario_MZN) AS receita,

    -- métricas de rotas
    SUM(r.Viagens_Realizadas)   AS viagens,
    SUM(r.Passageiros_Total)    AS passageiros,
    AVG(r.Tempo_Medio_Min)      AS duracao_min

FROM vendas v
JOIN segmentos s ON v.ID_Segmento = s.ID_Segmento
JOIN rotas     r ON v.ID_Rota = r.ID_Rota
                AND v.Data    = r.Data

GROUP BY
    v.Data,
    s.Tipo_Passageiro,
    (r.Origem || ' - ' || r.Destino),
    v.Forma_Pagamento;

In [ ]:
%%sql
-- Visualizar os primeiros registos — repara nos erros visíveis
-- Ex: Bilhetes_Vendidos = -460.0, receita = -34500.0, dia = '2026-03-01 00:00:00'
SELECT * FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS
LIMIT 10;

---
# 🔍 Detectar os Erros

**Regra de ouro:** Antes de corrigir, sempre **confirmar** quantos registos têm erro  
e **verificar** que a detecção é precisa (não apanha falsos positivos).

## ✅ Erro 1 — NEGATIVO: valores numéricos < 0

Os erros negativos foram injectados apenas em:
- `Bilhetes_Vendidos` (tabela vendas)
- `Preco_Unitario_MZN` (tabela vendas)

A coluna `receita` é calculada (`Bilhetes × Preço`), por isso fica negativa em cascata quando um dos dois é negativo.
`viagens`, `passageiros` e `duracao_min` **nunca tiveram negativos** — não precisam de ser corrigidas.

In [ ]:
%%sql
-- Detectar registos com Bilhetes OU Preço negativos
-- receita negativa é consequência, não erro independente
SELECT *
FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS
WHERE Bilhetes_Vendidos < 0
   OR Preco_Unitario_MZN < 0;

> **Dica para estudantes:** Deves ver exactamente **6 linhas** — as mesmas injectadas no Part 0.  
> Se vires mais ou menos, algo correu mal no JOIN acima.

## ✅ Erro 2 — ESPAÇOS: espaços extra em `Forma_Pagamento`

Valores como `' M-Pesa '` ou `' e-Mola '` parecem iguais ao olho,  
mas numa query `WHERE Forma_Pagamento = 'M-Pesa'` **não são encontrados**.

**Como detectar:** comparar o valor original com a versão limpa pelo `TRIM()`

```sql
-- ❌ ERRADO: LIKE '%  %' procura dois espaços consecutivos no meio da string
--           nunca vai encontrar ' M-Pesa ' porque o espaço é só no início/fim
WHERE Forma_Pagamento LIKE '%  %'

-- ✅ CORRECTO: compara o valor com a versão sem espaços nas pontas
WHERE Forma_Pagamento <> TRIM(Forma_Pagamento)
```

In [ ]:
%%sql
-- Detectar registos com espaços extra no método de pagamento
-- TRIM() remove espaços no início e no fim da string
SELECT *
FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS
WHERE Forma_Pagamento <> TRIM(Forma_Pagamento);

> **Dica para estudantes:** Deves ver **3 linhas** — nota que na coluna `Forma_Pagamento`  
> o valor parece normal mas tem espaços invisíveis nas pontas: `' e-Mola '`

## ✅ Erro 3 — DATA_ERRADA: datetime em vez de date pura

As datas com erro foram guardadas como `'2026-03-01 00:00:00'` em vez de `'2026-03-01'`.
Em SQLite, as datas são texto — por isso a detecção é feita por comprimento ou padrão:

```sql
-- ❌ ERRADO: CAST(dia AS DATE) em SQLite não faz o que se espera
--           SQLite não tem tipo DATE nativo — o CAST devolve o mesmo texto
--           por isso a comparação CAST(dia AS DATE) <> dia é sempre falsa
WHERE CAST(dia AS DATE) <> dia

-- ✅ CORRECTO: uma date pura tem exactamente 10 caracteres (YYYY-MM-DD)
--             se tem mais de 10, tem hora incluída
WHERE LENGTH(dia) > 10
```

In [ ]:
%%sql
-- Detectar registos onde a data inclui a hora (comprimento > 10 caracteres)
-- Uma date pura YYYY-MM-DD tem exactamente 10 caracteres
SELECT dia, LENGTH(dia) AS comprimento, segmento, rota
FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS
WHERE LENGTH(dia) > 10
LIMIT 10;

> **Dica para estudantes:** Deves ver linhas com `comprimento = 19` (`'2026-03-01 00:00:00'`).  
> Repara que **todas as linhas têm este problema** — porque quando se faz o JOIN,  
> o pandas converte todas as datas para datetime ao ler o Excel.  
> A limpeza com `DATE()` resolve isso de uma vez para todas as linhas.

---
# 🧹 Criar Tabela Limpa — `MOZ_DEV_MOZCHAPPA100_CLEAN`

Agora que confirmámos todos os erros, criamos uma tabela limpa aplicando as correcções:  
- `DATE(dia)` → remove a hora, fica só `YYYY-MM-DD`
- `TRIM(Forma_Pagamento)` → remove espaços nas pontas
- `ABS(Bilhetes_Vendidos)` e `ABS(Preco_Unitario_MZN)` → corrige os valores negativos
- `ABS(receita)` → corrige em cascata (era negativa por causa dos dois acima)

> **Nota importante:** `viagens`, `passageiros` e `duracao_min` **não usam `ABS()`**  
> porque nunca tiveram erros negativos. Aplicar `ABS()` sem necessidade é mau hábito —  
> podes estar a esconder erros reais futuros.

In [ ]:
%%sql
-- Criar tabela limpa com as 3 correcções aplicadas
CREATE TABLE MOZ_DEV_MOZCHAPPA100_CLEAN AS
SELECT
    -- CORRECÇÃO 3: DATE() remove a hora da datetime
    DATE(dia)                       AS dia,

    segmento,
    rota,

    -- CORRECÇÃO 2: TRIM() remove espaços nas pontas
    TRIM(Forma_Pagamento)           AS Forma_Pagamento,

    -- CORRECÇÃO 1: ABS() só nas colunas que tinham negativos
    ABS(Bilhetes_Vendidos)          AS Bilhetes_Vendidos,
    ABS(Preco_Unitario_MZN)         AS Preco_Unitario_MZN,
    ABS(receita)                    AS receita,

    -- estas colunas não tinham erros — sem ABS()
    viagens,
    passageiros,
    duracao_min

FROM MOZ_DEV_MOZCHAPPA100_STELA_NEIL_ANALYSIS;

## ✔️ Verificar — confirmar que os erros foram corrigidos

Após criar a tabela limpa, **re-executar as 3 queries de detecção** na nova tabela.  
Todas devem devolver **zero linhas**.

In [ ]:
%%sql
-- Verificação 1: não deve haver negativos
SELECT COUNT(*) AS negativos_restantes
FROM MOZ_DEV_MOZCHAPPA100_CLEAN
WHERE Bilhetes_Vendidos < 0
   OR Preco_Unitario_MZN < 0
   OR receita < 0;

In [ ]:
%%sql
-- Verificação 2: não deve haver espaços
SELECT COUNT(*) AS espacos_restantes
FROM MOZ_DEV_MOZCHAPPA100_CLEAN
WHERE Forma_Pagamento <> TRIM(Forma_Pagamento);

In [ ]:
%%sql
-- Verificação 3: todas as datas devem ter exactamente 10 caracteres
SELECT COUNT(*) AS datas_com_hora
FROM MOZ_DEV_MOZCHAPPA100_CLEAN
WHERE LENGTH(dia) > 10;

> ✅ **Resultado esperado:** os 3 COUNTs devem devolver `0`.  
> Se algum devolver > 0, a correcção correspondente não funcionou — revê a query de criação da tabela.

## 👁️ Visualizar tabela limpa final

In [ ]:
%%sql
-- Visualizar os primeiros registos da tabela limpa
-- Confirmar: dia sem hora, pagamento sem espaços, sem negativos
SELECT * FROM MOZ_DEV_MOZCHAPPA100_CLEAN
LIMIT 10;

---
## 💾 Exportar para Excel — `MozChapa100_CLEAN.xlsx`

Por fim, exportamos a tabela limpa para Excel.  
Este ficheiro é o input do passo seguinte (análise com STELA no Excel / dashboard).

In [ ]:
# Exportar tabela limpa para Excel
df = %sql SELECT * FROM MOZ_DEV_MOZCHAPPA100_CLEAN
df = df.DataFrame()

output_path = "MozChapa100_CLEAN.xlsx"
df.to_excel(output_path, index=False)

print(f"✅ Ficheiro exportado: {output_path}")
print(f"   Linhas: {len(df):,}")
print(f"   Colunas: {list(df.columns)}")

---
## 🏁 Resumo do que aprendemos neste notebook

| Passo | O que fizemos | Função SQL |
|-------|--------------|------------|
| 1 | Carregar Excel → SQLite | `pandas.to_sql()` |
| 2 | Juntar 3 tabelas | `JOIN ... ON` |
| 3 | Detectar negativos | `WHERE col < 0` |
| 4 | Detectar espaços | `WHERE col <> TRIM(col)` |
| 5 | Detectar datas com hora | `WHERE LENGTH(col) > 10` |
| 6 | Criar tabela limpa | `CREATE TABLE AS SELECT` com `ABS()`, `TRIM()`, `DATE()` |
| 7 | Verificar limpeza | `COUNT(*)` nas queries de detecção → devem dar 0 |
| 8 | Exportar para Excel | `df.to_excel()` |

> **Próximo passo → Part 1:** análise e criação do dashboard no Excel com STELA 📊